In [4]:
import os 
import pandas as pd

option_dir = r'C:\Users\kylec\OneDrive\Desktop\react_project\my-tutorial\vertiflow\public\data_dev\Project1\Direct-3Zone-DD-Lunch'

In [24]:
zones = [f for f in os.listdir(option_dir) if os.path.isdir(os.path.join(option_dir, f))]
for zone in zones:
    zone_dir = os.path.join(option_dir, zone)
    sims = [f for f in os.listdir(zone_dir) if os.path.isdir(os.path.join(zone_dir, f))]
    for sim in sims:
        sim_dir = os.path.join(zone_dir, sim)
        runs = [f for f in os.listdir(sim_dir) if os.path.isdir(os.path.join(sim_dir, f))]
        data = {
            "timeline_logbook": {},
        }
        for run in runs:
            run_dir = os.path.join(sim_dir, run)
            files = [f for f in os.listdir(run_dir) if f.endswith('.feather')]
            for f in files:
                path = os.path.join(run_dir, f)
                if f.startswith('timeline_logbook'):
                    lvlID = 'all' if len(f.split('_')) == 2 else f.split('_')[-1].split('.feather')[0]
                    df_timeline = pd.read_feather(path)
                    if run == 'compiled':
                        data["timeline_logbook"].setdefault(lvlID, {})['compiled'] = df_timeline
                    else:
                        data["timeline_logbook"].setdefault(lvlID, {}).setdefault('runs', []).append(df_timeline)
        
        for lvlID, timeline_dict in data["timeline_logbook"].items():
            df_template = timeline_dict['compiled'][['time']]
            awt_list = []
            att_list = []
            for df_run in timeline_dict['runs']:
                df_awt = df_template.copy()
                df_awt = df_awt.merge(df_run[['time', 'mean_wait_time']], on='time', how='left')
                df_att = df_template.copy()
                df_att = df_att.merge(df_run[['time', 'mean_transit_time']], on='time', how='left')
                awt_list.append(df_awt['mean_wait_time'])
                att_list.append(df_att['mean_transit_time'])
            df_awt = pd.concat(awt_list, axis=1)
            df_att = pd.concat(att_list, axis=1)
            awt_min = df_awt.apply(lambda row: row.min(), axis=1)
            awt_max = df_awt.apply(lambda row: row.max(), axis=1)
            att_min = df_att.apply(lambda row: row.min(), axis=1)
            att_max = df_att.apply(lambda row: row.max(), axis=1)

            df_timeline = timeline_dict['compiled']
            df_timeline['min_awt'] = awt_min
            df_timeline['max_awt'] = awt_max
            df_timeline['min_att'] = att_min
            df_timeline['max_att'] = att_max
            df_timeline.to_feather(os.path.join(sim_dir, 'compiled', f'timeline_logbook.feather'))
                

In [29]:
zones = [f for f in os.listdir(option_dir) if os.path.isdir(os.path.join(option_dir, f))]
for zone in zones:
    zone_dir = os.path.join(option_dir, zone)
    sims = [f for f in os.listdir(zone_dir) if os.path.isdir(os.path.join(zone_dir, f))]
    for sim in sims:
        sim_dir = os.path.join(zone_dir, sim)
        runs = [f for f in os.listdir(sim_dir) if os.path.isdir(os.path.join(sim_dir, f))]
        for run in runs:
            run_dir = os.path.join(sim_dir, run)
            files = [f for f in os.listdir(run_dir) if f.endswith('.feather')]
            for f in files:
                path = os.path.join(run_dir, f)
                if f.startswith('timeline_logbook') and run == 'compiled':
                    df_timeline = pd.read_feather(path)
                    # df_timeline.to_csv('C:\\Users\\kylec\\OneDrive\\Desktop\\react_project\\my-tutorial\\vertiflow\\public\\data_dev\\Project1\\Direct-3Zone-DD-Lunch\\South Tower - Office - High Zone\\726\\compiled\\timeline_logbook.csv')
                    # break
                    try:
                        df_timeline['min_awt_temp'] = df_timeline['mean_wait_time_register'].apply(lambda lst: min(lst) if len(lst)!= 0 else 0)
                        df_timeline['max_awt_temp'] = df_timeline['mean_wait_time_register'].apply(lambda lst: max(lst) if len(lst)!= 0 else 0)
                        if 'min_awt' in df_timeline.columns and not df_timeline['min_awt'].equals(df_timeline['min_awt_temp']):
                            print(f"Mismatch min_awt for {f} in {run} of {sim} in {zone}")
                            print(df_timeline['min_awt'].head(20), df_timeline['min_awt_temp'].head(20))
                        if 'max_awt' in df_timeline.columns and not df_timeline['max_awt'].equals(df_timeline['max_awt_temp']):
                            print(f"Mismatch max_awt for {f} in {run} of {sim} in {zone}")
                            print(df_timeline['max_awt'].head(20), df_timeline['max_awt_temp'].head(20))
                            break
                    except:
                        print(f, run, sim, zone)
                print("==============================")

                    